In [79]:
import torch
import torch.nn as nn
from deepspeech_pytorch.model import DeepSpeech

# local imports
from auditory_cortex import utils
from auditory_cortex import results_dir, pretrained_dir

In [115]:
class DeepSpeechsEncoder(torch.nn.Module):
    def __init__(self, model_name='deepspeech2', layer_id=0):
        super().__init__()
        self.model_name = model_name
        self.config = utils.load_dnn_config(model_name=self.model_name)
        self.layers_dict = {         # layer_id: layer_name
            layer['layer_id']:layer['layer_name']
            for layer in self.config['layers']
            }
        checkpoint = pretrained_dir / model_name / self.config['saved_checkpoint']
        self.model = DeepSpeech.load_from_checkpoint(checkpoint_path=checkpoint)

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)

        # self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False

        # setting RNN layers to train mode to enable full backward pass
        for m in self.model.modules():
            if isinstance(m, (nn.RNN, nn.LSTM, nn.GRU)):
                m.train()  # enable full backward pass
            else:
                m.eval()   # keep others in eval (e.g., batchnorm, dropout)

        self.register_hooks(layer_id)
        self.layer_features = None
    
    def get_layer_features(self, x):
        self.layer_features = None
        _ = self.forward(x)
        return self.layer_features

    def register_hooks(self, layer_id):
        layer_name = self.layers_dict[layer_id]
        for name, module in self.model.named_modules():
            if name == layer_name:
                module.__name__ = layer_name
                module.register_forward_hook(self.hook_fn)

    def hook_fn(self, module, input, output):
        if 'rnn' in module.__name__:
            features, seq_lengths = nn.utils.rnn.pad_packed_sequence(output[0], batch_first=True)
        else:
            # output = output.squeeze()
            if 'conv' in module.__name__:
                if output.ndim == 4:
                    output = output.reshape(output.shape[0], -1, output.shape[-1])
                output = output.transpose(1,2)
            elif 'coch' in self.model_name:
                if output.ndim > 2:
                    output = output.reshape(output.shape[0]*output.shape[1], -1)
                output = output.transpose(0,1)      # (time, features) format
                if output.shape[1] > 2000:
                    # restrict the output to max 2000 features
                    output = output[:, :2000]   
            features = output
        self.layer_features = features

    def forward(self, x):
        spect = self.process_input(x).unsqueeze(1)  # add channel dimension
        lengths = torch.full((spect.shape[0],), spect.shape[-1], dtype=torch.int64, device=self.device)
        out = self.model(spect, lengths)
        return out

    def process_input(self, y):
        """
        :param y: Audio signal as an array of float numbers
        :return: Spectrogram of the signal
        """
        sample_rate = self.config.get('sample_rate', 16000)
        window_size = self.config.get('window_size', 0.02)
        window_stride = self.config.get('window_stride', 0.01)
        window = self.config.get('window', 'hamming')
        normalize = self.config.get('normalize', True)

        n_fft = int(sample_rate * window_size)
        win_length = n_fft
        hop_length = int(sample_rate * window_stride)
        # STFT
        D = torch.stft(
            y, n_fft=n_fft, hop_length=hop_length, win_length=win_length,
            window=torch.hamming_window(win_length, device=self.device), 
            return_complex=True,
            pad_mode='constant', center=True, normalized=False, onesided=True 
            )
        spect = torch.abs(D)
        spect = torch.log1p(spect)
        if normalize:
            spect = (spect - spect.mean()) / (spect.std() + 1e-5)
        return spect




In [129]:
import librosa
wav = librosa.load("/home/ahmedb/projects/AudioLDM/trumpet.wav", sr=16000)[0]

x = torch.FloatTensor(wav).unsqueeze(0).cuda()

In [130]:
encoder = DeepSpeechsEncoder(layer_id=2)

Lightning automatically upgraded your loaded checkpoint from v1.1.5 to v2.0.8. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file ../../../../../depot/jgmakin/data/auditory_cortex/results/pretrained_weights/deepspeech2/librispeech_pretrained_v3.ckpt`


In [134]:
x.requires_grad = True
out = encoder.get_layer_features(x)

loss = out.sum()
loss.backward()

tensor(-45.8102, device='cuda:0', grad_fn=<SumBackward0>)

In [117]:
x = torch.cat([x, x, x], dim=0)

In [76]:
x = torch.randn(2, 10, 12, 180)

In [78]:
x.reshape(x.shape[0], -1, x.shape[-1]).shape

torch.Size([2, 120, 180])

In [118]:
x.shape

torch.Size([3, 47554])

In [84]:
x.shape

torch.Size([2, 47554])

In [123]:
encoder = DeepSpeechsEncoder(layer_id=1)
out = encoder.get_layer_features(x)
print(out.shape)

Lightning automatically upgraded your loaded checkpoint from v1.1.5 to v2.0.8. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file ../../../../../depot/jgmakin/data/auditory_cortex/results/pretrained_weights/deepspeech2/librispeech_pretrained_v3.ckpt`


torch.Size([3, 149, 1312])


In [124]:
encoder = DeepSpeechsEncoder(layer_id=2)
out = encoder.get_layer_features(x)
print(out.shape)

Lightning automatically upgraded your loaded checkpoint from v1.1.5 to v2.0.8. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file ../../../../../depot/jgmakin/data/auditory_cortex/results/pretrained_weights/deepspeech2/librispeech_pretrained_v3.ckpt`


torch.Size([3, 149, 2048])


In [125]:
encoder = DeepSpeechsEncoder(layer_id=3)
out = encoder.get_layer_features(x)
print(out.shape)

Lightning automatically upgraded your loaded checkpoint from v1.1.5 to v2.0.8. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file ../../../../../depot/jgmakin/data/auditory_cortex/results/pretrained_weights/deepspeech2/librispeech_pretrained_v3.ckpt`


torch.Size([3, 149, 2048])


In [126]:
encoder = DeepSpeechsEncoder(layer_id=4)
out = encoder.get_layer_features(x)
print(out.shape)

Lightning automatically upgraded your loaded checkpoint from v1.1.5 to v2.0.8. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file ../../../../../depot/jgmakin/data/auditory_cortex/results/pretrained_weights/deepspeech2/librispeech_pretrained_v3.ckpt`


torch.Size([3, 149, 2048])


In [127]:
encoder = DeepSpeechsEncoder(layer_id=5)
out = encoder.get_layer_features(x)
print(out.shape)

Lightning automatically upgraded your loaded checkpoint from v1.1.5 to v2.0.8. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file ../../../../../depot/jgmakin/data/auditory_cortex/results/pretrained_weights/deepspeech2/librispeech_pretrained_v3.ckpt`


torch.Size([3, 149, 2048])


In [128]:
encoder = DeepSpeechsEncoder(layer_id=6)
out = encoder.get_layer_features(x)
print(out.shape)

Lightning automatically upgraded your loaded checkpoint from v1.1.5 to v2.0.8. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file ../../../../../depot/jgmakin/data/auditory_cortex/results/pretrained_weights/deepspeech2/librispeech_pretrained_v3.ckpt`


torch.Size([3, 149, 2048])


In [120]:
out.shape

torch.Size([3, 149, 2048])

In [109]:
chunks = out[0].data.chunk(3, dim=0)
data = torch.stack(chunks, dim=0)

In [112]:
out, seq_lengths = nn.utils.rnn.pad_packed_sequence(out[0], batch_first=True)

In [113]:
out.shape

torch.Size([3, 149, 2048])

In [114]:
seq_lengths

tensor([149, 149, 149])

In [111]:
out[0].shape

AttributeError: 'PackedSequence' object has no attribute 'shape'

In [110]:
data.shape

torch.Size([3, 149, 2048])

In [106]:
chunks[0].shape

torch.Size([149, 2048])

In [105]:
len(chunks)

3

In [73]:
len(out[1])

2

In [74]:
out[1][0].shape

torch.Size([2, 2, 1024])

In [75]:
out[1][1].shape

torch.Size([2, 2, 1024])

torch.Size([2, 32, 81, 149])


In [32]:
out[0].shape

torch.Size([2, 149, 29])

In [27]:
out.shape

AttributeError: 'tuple' object has no attribute 'shape'

In [12]:
def register_hooks(model, layer_names):
    for name, module in model.named_modules():
        if name in layer_names:
            module.register_forward_hook(hook_fn)

def hook_fn(module, input, output):
    print(f"Inside {module.__class__.__name__} forward hook")
    print(f"Input shape: {input[0].shape}")
    print(f"Output shape: {output.shape}")



In [13]:
register_hooks(model, [layers_dict[0]])

In [9]:
pretrained_dir

PosixPath('/depot/jgmakin/data/auditory_cortex/results/pretrained_weights')

In [3]:
config['layers']

[{'layer_name': 'conv.seq_module.2',
  'layer_id': 0,
  'layer_type': 'conv',
  'RF': 120},
 {'layer_name': 'conv.seq_module.5',
  'layer_id': 1,
  'layer_type': 'conv',
  'RF': 320},
 {'layer_name': 'rnns.0.rnn', 'layer_id': 2, 'layer_type': 'rnn', 'RF': 320},
 {'layer_name': 'rnns.1.rnn', 'layer_id': 3, 'layer_type': 'rnn', 'RF': 0},
 {'layer_name': 'rnns.2.rnn', 'layer_id': 4, 'layer_type': 'rnn', 'RF': 0},
 {'layer_name': 'rnns.3.rnn', 'layer_id': 5, 'layer_type': 'rnn', 'RF': 0},
 {'layer_name': 'rnns.4.rnn', 'layer_id': 6, 'layer_type': 'rnn', 'RF': 0}]